# 02 — Split Sequences

This notebook:

1. Snaps Mapillary speed signs onto sequences (with heading validation)
2. Detects bearing-change turn points
3. Splits sequences into labeled sub-edges
4. Visualizes the results for parameter tuning

**Inputs:** signs + sequences from notebook 01 (or re-fetch here)

In [ ]:
import os
import geopandas as gpd
from slc import fetch, snap, split, viz

TOKEN = os.environ['MAPILLARY_ACCESS_TOKEN']
BBOX = (-111.920, 40.855, -111.855, 40.910)

In [ ]:
signs = fetch.fetch_mapillary_signs(BBOX, TOKEN)
images = fetch.fetch_mapillary_images(BBOX, TOKEN)
sequences = fetch.build_sequences(images)
overture_raw = fetch.fetch_overture_segments(BBOX)
overture = fetch.extract_overture_speed_limits(overture_raw)

In [ ]:
# Snap signs to sequences
snapped = snap.snap_signs_to_sequences(
    signs, sequences,
    max_distance_m=30.0,
    max_heading_diff=30.0,
)
print(f'Snapped sign-sequence pairs: {len(snapped)}')
snapped.head()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
snapped['snap_distance_m'].hist(bins=20)
plt.xlabel('Snap distance (m)')
plt.title('Distribution of snap distances')
plt.show()

print(f'Heading agreement rate: {snapped["heading_agreement"].mean():.1%}')

In [ ]:
# Split all sequences
split_edges = split.split_all_sequences(
    sequences, snapped,
    bearing_threshold_deg=60.0,
    window_m=80.0,
)
print(f'Split edges: {len(split_edges)}')
print(split_edges['speed_mph'].value_counts(dropna=False))

In [ ]:
# Export for next notebook
split_edges.to_parquet('split_edges.parquet', index=False)
print('Saved split_edges.parquet')

In [ ]:
# Visualize split edges
viz.map_split_edges(split_edges, signs=signs, overture=overture)